# 01 — Ingest Grid Data into Lakehouse and KQL Database
This notebook reads CSV files from the landing zone, writes them as Delta tables,
and also ingests them into the KQL Database for real-time analytics.

In [ ]:
# Read all CSV files from landing zone
df_sensors = spark.read.option("header", "true").option("inferSchema", "true").csv("{{DATA_SOURCE_PATH}}/grid_sensors.csv")
df_events = spark.read.option("header", "true").option("inferSchema", "true").csv("{{DATA_SOURCE_PATH}}/power_events.csv")
df_renewable = spark.read.option("header", "true").option("inferSchema", "true").csv("{{DATA_SOURCE_PATH}}/renewable_generation.csv")

print(f"Grid sensors: {df_sensors.count()} rows")
print(f"Power events: {df_events.count()} rows")
print(f"Renewable generation: {df_renewable.count()} rows")

In [ ]:
# Write all datasets as Delta tables in the Lakehouse
df_sensors.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable("grid_sensors")
df_events.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable("power_events")
df_renewable.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable("renewable_generation")
print("All Delta tables written to Lakehouse")

In [ ]:
# Ingest into KQL Database for real-time analytics
KUSTO_CLUSTER = "{{EVENTHOUSE_URI}}"
KUSTO_DB = "{{KQL_DATABASE_NAME}}"

# Skip KQL ingestion if Eventhouse URI is not configured
if KUSTO_CLUSTER and not KUSTO_CLUSTER.startswith("{{"):
    try:
        access_token = mssparkutils.credentials.getToken("kusto")
        print(f"Ingesting into KQL: {KUSTO_CLUSTER} / {KUSTO_DB}")
    except Exception as e:
        print(f"Could not get Kusto token: {e}")
        access_token = None
else:
    print("No Eventhouse URI configured — skipping KQL ingestion")
    access_token = None

In [ ]:
if access_token:
    # Ingest grid sensors
    df_sensors.write.format("com.microsoft.kusto.spark.synapse.datasource") \
        .option("kustoCluster", KUSTO_CLUSTER) \
        .option("kustoDatabase", KUSTO_DB) \
        .option("kustoTable", "GridSensors") \
        .option("accessToken", access_token) \
        .option("tableCreateOptions", "CreateIfNotExist") \
        .mode("Append") \
        .save()
    print("GridSensors ingested")

    # Ingest power events
    df_events.write.format("com.microsoft.kusto.spark.synapse.datasource") \
        .option("kustoCluster", KUSTO_CLUSTER) \
        .option("kustoDatabase", KUSTO_DB) \
        .option("kustoTable", "PowerEvents") \
        .option("accessToken", access_token) \
        .option("tableCreateOptions", "CreateIfNotExist") \
        .mode("Append") \
        .save()
    print("PowerEvents ingested")

    # Ingest renewable generation
    df_renewable.write.format("com.microsoft.kusto.spark.synapse.datasource") \
        .option("kustoCluster", KUSTO_CLUSTER) \
        .option("kustoDatabase", KUSTO_DB) \
        .option("kustoTable", "RenewableGeneration") \
        .option("accessToken", access_token) \
        .option("tableCreateOptions", "CreateIfNotExist") \
        .mode("Append") \
        .save()
    print("RenewableGeneration ingested")
    print("All data ingested into KQL Database!")
else:
    print("KQL ingestion skipped — data is available in Lakehouse Delta tables")